# ICU Early Warning System — Deterioration Prediction

**Pipeline:** Problem Definition -> Data Collection -> Cleaning -> EDA -> Feature Engineering ->
Train/Test Split -> Model Selection -> Training -> Evaluation -> Hyperparameter Tuning ->
Final Testing -> Explainability -> Export Artifacts -> Monitoring (design)

**Dataset:** MIMIC-III Clinical Database Demo (~100 patients, public, no credentialing required).
In production this pipeline would run against full MIMIC-IV.

### Table of Contents
- [0. Setup](#0)
- [1. Problem Definition](#1)
- [2. Data Collection](#2)
- [3. Data Cleaning & Preparation](#3)
- [4. Exploratory Data Analysis](#4)
- [5. Feature Engineering](#5)
- [6. Train/Test Split](#6)
- [7. Model Selection & Training](#7)
- [8. Model Evaluation](#8)
- [9. Hyperparameter Tuning](#9)
- [10. Final Testing](#10)
- [11. Explainability (SHAP)](#11)
- [12. Export Artifacts](#12)
- [13. Monitoring & Retraining (Design)](#13)
- [How to Run](#run)


<a id='0'></a>
## 0. Setup

In [1]:
# --- Pinned installs (comment out versions if Colab conflicts force a restart) ---
!pip install -q xgboost==2.0.3 shap==0.45.0 scikit-learn==1.5.0


OSError: [WinError 1450] Insufficient system resources exist to complete the requested service

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [279 lines of output]
      + meson setup C:\Users\chandrajit\AppData\Local\Temp\pip-install-on02e53_\scikit-learn_905beeeb0c724516a878bc5a65996797 C:\Users\chandrajit\AppData\Local\Temp\pip-install-on02e53_\scikit-learn_905beeeb0c724516a878bc5a65996797\.mesonpy-05mggotg -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\chandrajit\AppData\Local\Temp\pip-install-on02e53_\scikit-learn_905beeeb0c724516a878bc5a65996797\.mesonpy-05mggotg\meson-python-native-file.ini
      The Meson build system
      Version: 1.12.0
      Source dir: C:\Users\chandrajit\AppData\Local\Temp\pip-install-on02e53_\scikit-learn_905beeeb0c724516a878bc5a65996797
      Build dir: C:\Users\chandrajit\AppData\Local\Temp\pip-install-on02e53_\scikit-learn_905beeeb0c724516a878bc5a65996797\.mesonpy-05mggotg
      Build type: native build
      Activating VS 17.14.37 

In [ ]:
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    f1_score, recall_score, precision_score, brier_score_loss, confusion_matrix
)
from xgboost import XGBClassifier
import joblib
import shap

warnings.filterwarnings("ignore")

# --- Reproducibility ---
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

sns.set_style("whitegrid")
print("Setup complete. Random seed fixed at", RANDOM_SEED)


**Mount Google Drive** — data and model artifacts persist here across Colab sessions.

In [ ]:
# Detect environment (Google Colab vs. Local)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/icu_early_warning'
except Exception:
    import os
    PROJECT_DIR = os.path.abspath('.')

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
ARTIFACT_DIR = os.path.join(PROJECT_DIR, 'artifacts')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print("Project dir:", PROJECT_DIR)
print("Data dir:   ", DATA_DIR)
print("Artifact dir:", ARTIFACT_DIR)


**Download MIMIC-III Demo from PhysioNet.**

Requires a free PhysioNet account with the MIMIC-III Demo data use terms accepted
(no CITI training / credentialing needed for the demo set).

This cell will prompt for your PhysioNet username/password on first run.
Skips re-download if files already exist in Drive.

In [ ]:
# Verify local MIMIC-III demo CSV files
expected_files = [
    "PATIENTS.csv", "ADMISSIONS.csv", "ICUSTAYS.csv",
    "CHARTEVENTS.csv", "LABEVENTS.csv", "D_ITEMS.csv", "D_LABITEMS.csv"
]

missing_files = [f for f in expected_files if not os.path.exists(os.path.join(DATA_DIR, f))]
if not missing_files:
    print(f"All {len(expected_files)} MIMIC-III Demo tables verified in {DATA_DIR}!")
else:
    print(f"Warning: Missing files: {missing_files}. Please extract from mimic-iii-clinical-database-demo-1.4.zip.")


<a id='1'></a>
## 1. Problem Definition

**Target event (composite deterioration):** death, new vasopressor initiation, new
intubation/mechanical ventilation, or cardiac arrest within the prediction horizon.

**Prediction horizon (H):** 6 hours.

**Observation window (W):** past 4-8 hours of vitals/labs.

**Unit of prediction:** per patient, per hour (sliding window over the ICU stay).

**Metrics:**
- AUROC, PR-AUC (discrimination)
- F1, Recall, Precision at chosen operating threshold
- Calibration: Brier score / reliability plot
- Clinical utility: lead time (hours of advance warning before a true event), % events warned >=4h / >=6h ahead

**Flow:**
Problem Definition -> Data Collection -> Cleaning -> EDA -> Feature Engineering ->
Split -> Model Selection -> Training -> Evaluation -> Tuning -> Final Testing ->
Explainability -> Deployment -> Monitoring

**Known limitation flagged early:** lab measurement frequency can itself leak clinical
suspicion (informative missingness) — addressed in Section 5.


<a id='2'></a>
## 2. Data Collection

**Data dictionary (tables used):**
- `PATIENTS` — demographics (subject_id, gender, dob)
- `ADMISSIONS` — hospital admission info (admission type, hadm_id)
- `ICUSTAYS` — ICU stay boundaries (icustay_id, intime, outtime)
- `CHARTEVENTS` — vitals time series (itemid, charttime, valuenum)
- `LABEVENTS` — lab results time series (itemid, charttime, valuenum)
- `D_ITEMS` — lookup table mapping chartevents itemid -> variable name
- `D_LABITEMS` — lookup table mapping labevents itemid -> variable name

**Note:** This is the MIMIC-III **Demo** dataset (~100 patients). Results here are for
pipeline validation only; in production this would run against full, credentialed MIMIC-IV.

In [ ]:
patients = pd.read_csv(os.path.join(DATA_DIR, "PATIENTS.csv"))
admissions = pd.read_csv(os.path.join(DATA_DIR, "ADMISSIONS.csv"))
icustays = pd.read_csv(os.path.join(DATA_DIR, "ICUSTAYS.csv"))
chartevents = pd.read_csv(os.path.join(DATA_DIR, "CHARTEVENTS.csv"))
labevents = pd.read_csv(os.path.join(DATA_DIR, "LABEVENTS.csv"))
d_items = pd.read_csv(os.path.join(DATA_DIR, "D_ITEMS.csv"))
d_labitems = pd.read_csv(os.path.join(DATA_DIR, "D_LABITEMS.csv"))

for name, df in [("patients", patients), ("admissions", admissions), ("icustays", icustays),
                  ("chartevents", chartevents), ("labevents", labevents),
                  ("d_items", d_items), ("d_labitems", d_labitems)]:
    print(f"{name}: {df.shape}")


In [ ]:
icustays.head()


<a id='3'></a>
## 3. Data Cleaning & Preparation

Cohort filters: age >= 18, first ICU stay per patient, stay length >= 6 hours.
Physiologically impossible values are clipped/dropped per variable-specific bounds.
All timestamps converted to hours since ICU admission (`icustays.intime` = time zero).

In [ ]:
# --- Physiologic plausible bounds (clip outside these, drop if still absurd) ---
PHYSIO_BOUNDS = {
    "heart_rate":       (20, 250),   # bpm
    "resp_rate":        (4, 60),     # breaths/min
    "spo2":             (50, 100),   # %
    "sbp":              (40, 250),   # mmHg
    "dbp":              (20, 180),   # mmHg
    "temp_c":           (25, 43),    # Celsius
    "lactate":          (0.1, 30),   # mmol/L
    "creatinine":       (0.1, 20),   # mg/dL
}

def clip_physio(series, var_name):
    lo, hi = PHYSIO_BOUNDS.get(var_name, (series.min(), series.max()))
    before_n = series.notna().sum()
    clipped = series.clip(lower=lo, upper=hi)
    out_of_range = ((series < lo) | (series > hi)).sum()
    return clipped, out_of_range, before_n

print("Physiologic bounds defined for:", list(PHYSIO_BOUNDS.keys()))


In [ ]:
# --- Cohort filtering ---
icustays['intime'] = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
icustays['los_hours'] = (icustays['outtime'] - icustays['intime']).dt.total_seconds() / 3600

n_before_stays = icustays['icustay_id'].nunique()
n_before_patients = icustays['subject_id'].nunique()

# first ICU stay per subject
icustays_sorted = icustays.sort_values(['subject_id', 'intime'])
first_stay = icustays_sorted.groupby('subject_id').first().reset_index()

# merge age at admission (approx via patients.dob vs admission time)
patients['dob'] = pd.to_datetime(patients['dob'])
cohort = first_stay.merge(patients[['subject_id', 'dob', 'gender']], on='subject_id', how='left')
# Safe age calculation to prevent pandas 2.0+ int64 overflow from ~300yr obfuscated birthdates:
cohort['age'] = cohort['intime'].dt.year - cohort['dob'].dt.year
# MIMIC obfuscates ages >89 as ~300; treat those as 90+
cohort.loc[cohort['age'] > 89, 'age'] = 90.0

cohort = cohort[(cohort['age'] >= 18) & (cohort['los_hours'] >= 6)].copy()

n_after_stays = cohort['icustay_id'].nunique()
n_after_patients = cohort['subject_id'].nunique()

summary = pd.DataFrame({
    "metric": ["patients", "icu_stays"],
    "before_filter": [n_before_patients, n_before_stays],
    "after_filter": [n_after_patients, n_after_stays]
})
print(summary)
cohort_stays = cohort[['subject_id', 'hadm_id', 'icustay_id', 'intime', 'outtime',
                        'los_hours', 'age', 'gender']].copy()
cohort_stays.head()


In [ ]:
# --- Convert vitals/labs timestamps to hours-since-admission, restrict to cohort ---
chartevents['charttime'] = pd.to_datetime(chartevents['charttime'])
labevents['charttime'] = pd.to_datetime(labevents['charttime'])

cohort_vitals = chartevents.merge(
    cohort_stays[['icustay_id', 'intime']], on='icustay_id', how='inner'
)
cohort_vitals['hours_since_admit'] = (
    cohort_vitals['charttime'] - cohort_vitals['intime']
).dt.total_seconds() / 3600
cohort_vitals = cohort_vitals[cohort_vitals['hours_since_admit'] >= 0]

cohort_labs = labevents.merge(
    cohort_stays[['subject_id', 'hadm_id', 'icustay_id', 'intime']],
    on=['subject_id', 'hadm_id'], how='inner'
)
cohort_labs['hours_since_admit'] = (
    cohort_labs['charttime'] - cohort_labs['intime']
).dt.total_seconds() / 3600
cohort_labs = cohort_labs[cohort_labs['hours_since_admit'] >= 0]

print("cohort_vitals:", cohort_vitals.shape)
print("cohort_labs:", cohort_labs.shape)

pct_dropped_vitals = 100 * (1 - len(cohort_vitals) / len(chartevents))
pct_dropped_labs = 100 * (1 - len(cohort_labs) / len(labevents))
print(f"% chartevents rows dropped by cohort filter: {pct_dropped_vitals:.1f}%")
print(f"% labevents rows dropped by cohort filter: {pct_dropped_labs:.1f}%")


<a id='4'></a>
## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(cohort_stays['age'], bins=20, color='steelblue')
axes[0].set_title("Age distribution")
axes[1].hist(cohort_stays['los_hours'] / 24, bins=20, color='seagreen')
axes[1].set_title("ICU length of stay (days)")
axes[2].bar(['Male', 'Female'],
            cohort_stays['gender'].value_counts().reindex(['M', 'F']).values,
            color=['cornflowerblue', 'salmon'])
axes[2].set_title("Gender distribution")
plt.tight_layout()
plt.show()


**Interpretation:** _fill in after running — e.g., "median LOS is X days; cohort skews toward
older adults, consistent with typical ICU demographics."_

In [ ]:
# Missingness heatmap and class-balance/outcome-rate plots need the itemid->variable
# mapping and windows_df, both built in Section 5. See Section 5.1b and 5.2c below for
# those visuals — kept there so the code that produces the data sits next to the plot.
print("See Section 5.1b (class balance) and Section 5.2c (missingness heatmap) for these plots.")


<a id='5'></a>
## 5. Feature Engineering

### 5.1 Define windows and labels
For each ICU stay, create hourly time points `t` from hour 4 to `LOS - H`.
Label = 1 if a deterioration event occurs in `(t, t+H]`, else 0.

**Deterioration event proxy (demo dataset):** in-hospital death (`ADMISSIONS.deathtime`)
falling within the horizon. Vasopressor/intubation flags require `INPUTEVENTS`/`PROCEDUREEVENTS`
tables — add if available in your MIMIC extract; omitted here to keep the demo pipeline runnable
end-to-end with the core tables only.

**Leakage note:** measurement-count features (e.g., number of lactate draws in the window) can
reflect clinical suspicion rather than pure physiology — an informative-missingness leak. We
retain these features (standard practice) but flag this explicitly as a known limitation when
interpreting performance.

In [ ]:
H = 6   # prediction horizon, hours
W = 6   # observation window, hours

admissions['deathtime'] = pd.to_datetime(admissions['deathtime'])
cohort_stays = cohort_stays.merge(
    admissions[['hadm_id', 'deathtime']], on='hadm_id', how='left'
)
cohort_stays['death_hours_since_admit'] = (
    cohort_stays['deathtime'] - cohort_stays['intime']
).dt.total_seconds() / 3600

rows = []
for _, stay in cohort_stays.iterrows():
    max_t = stay['los_hours'] - H
    if max_t < 4:
        continue
    for t in np.arange(4, max_t, 1.0):
        label = 0
        if pd.notna(stay['death_hours_since_admit']):
            if t < stay['death_hours_since_admit'] <= t + H:
                label = 1
        rows.append({
            'icustay_id': stay['icustay_id'],
            'subject_id': stay['subject_id'],
            't': t,
            'label': label
        })

windows_df = pd.DataFrame(rows)
print("windows_df shape:", windows_df.shape)
print("Positive rate:", windows_df['label'].mean().round(4))
windows_df.head()


### 5.1b Class balance & time-to-event visuals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Class balance bar chart ---
label_counts = windows_df['label'].value_counts().reindex([0, 1])
bars = axes[0].bar(['No deterioration\n(0)', 'Deterioration\n(1)'], label_counts.values,
                    color=['#028090', '#F96167'])
axes[0].set_title("Class balance across all windows")
axes[0].set_ylabel("Number of windows")
for bar, val in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val, f'{val}\n({val/len(windows_df):.1%})',
                 ha='center', va='bottom', fontsize=10)

# --- Outcome rate over time-since-admission (hour t) ---
rate_by_hour = windows_df.groupby(windows_df['t'].round().astype(int))['label'].mean()
axes[1].plot(rate_by_hour.index, rate_by_hour.values, color='#F96167', marker='o', markersize=3)
axes[1].set_title("Positive rate by hour since admission")
axes[1].set_xlabel("Hour since ICU admission (t)")
axes[1].set_ylabel("Fraction of windows labeled positive")

# --- Time-to-event distribution: hours from admission to death, for patients who died ---
time_to_event = cohort_stays.loc[cohort_stays['death_hours_since_admit'].notna(),
                                  'death_hours_since_admit']
if len(time_to_event) > 0:
    axes[2].hist(time_to_event, bins=15, color='#990011')
    axes[2].set_title(f"Time-to-death distribution (n={len(time_to_event)})")
    axes[2].set_xlabel("Hours from ICU admission to death")
else:
    axes[2].text(0.5, 0.5, "No deaths in this demo cohort", ha='center', va='center')
    axes[2].set_title("Time-to-death distribution")

plt.tight_layout()
plt.show()

print(f"Positive windows: {label_counts.get(1, 0)} / {len(windows_df)} "
      f"({windows_df['label'].mean():.1%})")


### 5.2 Aggregate vitals into features (per window)

**Note:** `labevents` is loaded in Section 2 but lab features are built separately in
5.2b below — both vitals and labs feed the final feature set.

In [ ]:
# Map chartevents itemid -> readable variable name via d_items (adjust itemid list to your extract)
# Example itemid set for common MIMIC-III vitals (verify against d_items in your version):
VITAL_ITEMIDS = {
    211: 'heart_rate', 220045: 'heart_rate',
    618: 'resp_rate', 220210: 'resp_rate',
    646: 'spo2', 220277: 'spo2',
    51: 'sbp', 220050: 'sbp',
    8368: 'dbp', 220051: 'dbp',
    678: 'temp_f', 223761: 'temp_c',
}
cohort_vitals['variable'] = cohort_vitals['itemid'].map(VITAL_ITEMIDS)
cohort_vitals_mapped = cohort_vitals.dropna(subset=['variable']).copy()

# --- Convert Fahrenheit to Celsius, then collapse both into a single temp_c variable ---
is_f = cohort_vitals_mapped['variable'] == 'temp_f'
cohort_vitals_mapped.loc[is_f, 'valuenum'] = (cohort_vitals_mapped.loc[is_f, 'valuenum'] - 32) * 5 / 9
cohort_vitals_mapped.loc[is_f, 'variable'] = 'temp_c'

# --- Apply physiologic clipping now that variable names exist ---
clip_log = []
for var_name in cohort_vitals_mapped['variable'].unique():
    mask = cohort_vitals_mapped['variable'] == var_name
    clipped, n_out_of_range, n_total = clip_physio(cohort_vitals_mapped.loc[mask, 'valuenum'], var_name)
    cohort_vitals_mapped.loc[mask, 'valuenum'] = clipped
    clip_log.append({'variable': var_name, 'n_values': n_total, 'n_clipped': n_out_of_range})

clip_log_df = pd.DataFrame(clip_log)
print("Physiologic clipping applied. Out-of-range counts per variable:")
clip_log_df

def aggregate_window(icustay_id, t, w=W):
    window_data = cohort_vitals_mapped[
        (cohort_vitals_mapped['icustay_id'] == icustay_id) &
        (cohort_vitals_mapped['hours_since_admit'] > t - w) &
        (cohort_vitals_mapped['hours_since_admit'] <= t)
    ]
    feats = {}
    all_vars = set(VITAL_ITEMIDS.values()) - {'temp_f'}  # temp_f was collapsed into temp_c above
    for var in all_vars:
        sub = window_data[window_data['variable'] == var].dropna(subset=['valuenum', 'hours_since_admit'])
        vals = sub['valuenum']
        if len(vals) == 0:
            feats[f'{var}_last'] = np.nan
            feats[f'{var}_mean'] = np.nan
            feats[f'{var}_min'] = np.nan
            feats[f'{var}_max'] = np.nan
            feats[f'{var}_std'] = np.nan
            feats[f'{var}_slope'] = np.nan
            feats[f'{var}_count'] = 0
            feats[f'{var}_time_since_last'] = np.nan
        else:
            feats[f'{var}_last'] = vals.iloc[-1]
            feats[f'{var}_mean'] = vals.mean()
            feats[f'{var}_min'] = vals.min()
            feats[f'{var}_max'] = vals.max()
            feats[f'{var}_std'] = vals.std()
            x_vals = sub['hours_since_admit'].values
            if len(vals) >= 2 and (x_vals[-1] > x_vals[0]):
                slope = float(np.polyfit(x_vals, vals.values, 1)[0])
            else:
                slope = 0.0
            feats[f'{var}_slope'] = slope
            feats[f'{var}_count'] = len(vals)
            feats[f'{var}_time_since_last'] = t - sub['hours_since_admit'].iloc[-1]
    return feats

# NOTE: For the full demo run, apply aggregate_window() across all windows_df rows.
# This is O(n_windows) database-style lookups; for the ~100-patient demo it should
# complete in a few minutes. Vectorize/precompute per-stay if scaling to full MIMIC-IV.
print("aggregate_window() defined. Apply across windows_df next.")


In [ ]:
# Pre-indexed, high-speed vital aggregation
vitals_by_stay = {k: v.sort_values('hours_since_admit') for k, v in cohort_vitals_mapped.groupby('icustay_id')}
vital_vars = sorted(set(VITAL_ITEMIDS.values()) - {'temp_f'})

feature_rows = []
for row in windows_df.itertuples():
    stay_id, t = row.icustay_id, row.t
    feats = {'icustay_id': stay_id, 't': t}
    v_data = vitals_by_stay.get(stay_id)
    if v_data is not None and not v_data.empty:
        w_vitals = v_data[(v_data['hours_since_admit'] > t - W) & (v_data['hours_since_admit'] <= t)]
    else:
        w_vitals = None
        
    for var in vital_vars:
        if w_vitals is not None and not w_vitals.empty:
            sub = w_vitals[w_vitals['variable'] == var].dropna(subset=['valuenum', 'hours_since_admit'])
            vals = sub['valuenum']
        else:
            sub = pd.DataFrame()
            vals = pd.Series([], dtype=float)
            
        if len(vals) == 0:
            feats[f'{var}_last'] = np.nan
            feats[f'{var}_mean'] = np.nan
            feats[f'{var}_min'] = np.nan
            feats[f'{var}_max'] = np.nan
            feats[f'{var}_std'] = np.nan
            feats[f'{var}_slope'] = np.nan
            feats[f'{var}_count'] = 0
            feats[f'{var}_time_since_last'] = np.nan
        else:
            feats[f'{var}_last'] = float(vals.iloc[-1])
            feats[f'{var}_mean'] = float(vals.mean())
            feats[f'{var}_min'] = float(vals.min())
            feats[f'{var}_max'] = float(vals.max())
            feats[f'{var}_std'] = float(vals.std()) if len(vals) > 1 else 0.0
            x_vals = sub['hours_since_admit'].values
            if len(vals) >= 2 and (x_vals[-1] > x_vals[0]):
                feats[f'{var}_slope'] = float(np.polyfit(x_vals, vals.values, 1)[0])
            else:
                feats[f'{var}_slope'] = 0.0
            feats[f'{var}_count'] = len(vals)
            feats[f'{var}_time_since_last'] = float(t - sub['hours_since_admit'].iloc[-1])
            
    feature_rows.append(feats)

features_df = pd.DataFrame(feature_rows)
windows_df = windows_df.merge(features_df, on=['icustay_id', 't'], how='left')
print("windows_df with vital features:", windows_df.shape)


### 5.2c Missingness heatmap (per patient x vital, plus by hour-of-stay)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Patient x variable: how many measurements did each patient have for each vital?
vital_vars = sorted(set(VITAL_ITEMIDS.values()) - {'temp_f'})
measurement_counts = cohort_vitals_mapped.pivot_table(
    index='icustay_id', columns='variable', values='valuenum', aggfunc='count'
).reindex(columns=vital_vars).fillna(0)

sns.heatmap(measurement_counts.isna() | (measurement_counts == 0), cbar=False,
            cmap=['#028090', '#F2F2F2'], ax=axes[0], yticklabels=False)
axes[0].set_title("Missing vitals by patient (colored = never measured)")
axes[0].set_xlabel("Variable")
axes[0].set_ylabel("ICU stay")

# Variable x hour-of-stay bucket: how sparse is each vital over time?
cohort_vitals_mapped['hour_bucket'] = cohort_vitals_mapped['hours_since_admit'].astype(int)
hourly_counts = cohort_vitals_mapped.pivot_table(
    index='variable', columns='hour_bucket', values='valuenum', aggfunc='count'
).reindex(index=vital_vars).fillna(0)
hourly_counts = hourly_counts.loc[:, hourly_counts.columns <= 48]  # first 48h for readability

sns.heatmap(hourly_counts, cmap='viridis', ax=axes[1], cbar_kws={'label': '# measurements'})
axes[1].set_title("Measurement density by hour since admission (first 48h)")
axes[1].set_xlabel("Hour since admission")
axes[1].set_ylabel("Variable")

plt.tight_layout()
plt.show()


### 5.2b Aggregate labs into features (per window)

Verify these itemids against your `D_LABITEMS.csv` — lab itemids are consistent across
MIMIC-III versions but always confirm before trusting the mapping.

In [ ]:
# Pre-indexed, high-speed lab aggregation
labs_by_stay = {k: v.sort_values('hours_since_admit') for k, v in cohort_labs_mapped.groupby('icustay_id')}
lab_vars = sorted(set(LAB_ITEMIDS.values()))

lab_feature_rows = []
for row in windows_df.itertuples():
    stay_id, t = row.icustay_id, row.t
    feats = {'icustay_id': stay_id, 't': t}
    l_data = labs_by_stay.get(stay_id)
    if l_data is not None and not l_data.empty:
        w_labs = l_data[(l_data['hours_since_admit'] > t - W) & (l_data['hours_since_admit'] <= t)]
    else:
        w_labs = None
        
    for var in lab_vars:
        if w_labs is not None and not w_labs.empty:
            sub = w_labs[w_labs['variable'] == var].dropna(subset=['valuenum', 'hours_since_admit'])
            vals = sub['valuenum']
        else:
            vals = pd.Series([], dtype=float)
            
        if len(vals) == 0:
            feats[f'{var}_last'] = np.nan
            feats[f'{var}_mean'] = np.nan
            feats[f'{var}_min'] = np.nan
            feats[f'{var}_max'] = np.nan
            feats[f'{var}_count'] = 0
            feats[f'{var}_time_since_last'] = np.nan
        else:
            feats[f'{var}_last'] = float(vals.iloc[-1])
            feats[f'{var}_mean'] = float(vals.mean())
            feats[f'{var}_min'] = float(vals.min())
            feats[f'{var}_max'] = float(vals.max())
            feats[f'{var}_count'] = len(vals)
            feats[f'{var}_time_since_last'] = float(t - sub['hours_since_admit'].iloc[-1])
            
    lab_feature_rows.append(feats)

lab_features_df = pd.DataFrame(lab_feature_rows)
windows_df = windows_df.merge(lab_features_df, on=['icustay_id', 't'], how='left')
print("windows_df with vitals + labs:", windows_df.shape)


### 5.3 Add demographic / context features

In [ ]:
windows_df = windows_df.merge(
    cohort_stays[['icustay_id', 'age', 'gender']], on='icustay_id', how='left'
)
windows_df['gender_male'] = (windows_df['gender'] == 'M').astype(int)
windows_df.drop(columns=['gender'], inplace=True)


### 5.4 Compute NEWS2 score (clinical baseline + optional feature)

In [ ]:
def compute_news2(row):
    """Simplified NEWS2 score from resp rate, SpO2, temp, SBP, HR.
    Excludes consciousness/O2-supplementation sub-scores (not reliably available in demo tables).
    """
    score = 0
    rr = row.get('resp_rate_last', np.nan)
    if pd.notna(rr):
        if rr <= 8 or rr >= 25: score += 3
        elif 21 <= rr <= 24: score += 2
        elif 9 <= rr <= 11: score += 1

    spo2 = row.get('spo2_last', np.nan)
    if pd.notna(spo2):
        if spo2 <= 91: score += 3
        elif 92 <= spo2 <= 93: score += 2
        elif 94 <= spo2 <= 95: score += 1

    temp = row.get('temp_c_last', np.nan)
    if pd.notna(temp):
        if temp <= 35.0: score += 3
        elif temp >= 39.1: score += 2
        elif 35.1 <= temp <= 36.0 or 38.1 <= temp <= 39.0: score += 1

    sbp = row.get('sbp_last', np.nan)
    if pd.notna(sbp):
        if sbp <= 90 or sbp >= 220: score += 3
        elif 91 <= sbp <= 100: score += 2
        elif 101 <= sbp <= 110: score += 1

    hr = row.get('heart_rate_last', np.nan)
    if pd.notna(hr):
        if hr <= 40 or hr >= 131: score += 3
        elif 111 <= hr <= 130: score += 2
        elif (41 <= hr <= 50) or (91 <= hr <= 110): score += 1

    return score

windows_df['news2_score'] = windows_df.apply(compute_news2, axis=1)
print(windows_df['news2_score'].describe())


In [ ]:
feature_cols = [c for c in windows_df.columns
                if c not in ['icustay_id', 'subject_id', 't', 'label']]
print(f"Number of features: {len(feature_cols)}")
with open(os.path.join(ARTIFACT_DIR, 'feature_names.json'), 'w') as f:
    json.dump(feature_cols, f, indent=2)
feature_cols


<a id='6'></a>
## 6. Train/Test Split

Split by `subject_id` (patient-level), not by row, to avoid leakage across windows
belonging to the same patient. 70% train / 15% val / 15% test.

**Correction from review:** `GroupShuffleSplit` alone only prevents patient leakage — it
does **not** guarantee matched positive rates across splits. On a ~100-patient demo cohort
this matters a lot (a few unlucky patients can swing the split). We do a proper patient-level
**stratified group split**: label each patient by whether they *ever* had a positive window,
then stratify the patient-level split on that flag.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# One row per patient: did this patient ever have a positive window?
patient_outcome = windows_df.groupby('subject_id')['label'].max().reset_index()
patient_outcome.columns = ['subject_id', 'ever_positive']

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_SEED)
train_pat_idx, temp_pat_idx = next(sss1.split(patient_outcome, patient_outcome['ever_positive']))
train_patients = patient_outcome.iloc[train_pat_idx]['subject_id']
temp_patients_df = patient_outcome.iloc[temp_pat_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_SEED)
val_pat_idx, test_pat_idx = next(sss2.split(temp_patients_df, temp_patients_df['ever_positive']))
val_patients = temp_patients_df.iloc[val_pat_idx]['subject_id']
test_patients = temp_patients_df.iloc[test_pat_idx]['subject_id']

train_df = windows_df[windows_df['subject_id'].isin(train_patients)]
val_df = windows_df[windows_df['subject_id'].isin(val_patients)]
test_df = windows_df[windows_df['subject_id'].isin(test_patients)]

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    n_patients = df['subject_id'].nunique()
    n_windows = len(df)
    pos_rate = df['label'].mean()
    print(f"{name}: {n_patients} patients, {n_windows} windows, positive rate = {pos_rate:.4f}")

# sanity check: no patient overlap
assert set(train_df['subject_id']) & set(val_df['subject_id']) == set()
assert set(train_df['subject_id']) & set(test_df['subject_id']) == set()
assert set(val_df['subject_id']) & set(test_df['subject_id']) == set()
print("No patient overlap across splits — leakage check passed.")


In [ ]:
X_train, y_train = train_df[feature_cols], train_df['label']
X_val, y_val = val_df[feature_cols], val_df['label']
X_test, y_test = test_df[feature_cols], test_df['label']


<a id='7'></a>
## 7. Model Selection & Training

Three models, compared like-for-like on continuous risk scores:
1. **NEWS2** — rule-based clinical baseline (no training).
2. **Logistic Regression** — simple ML baseline (imputed + scaled features).
3. **XGBoost** — primary ML model (handles NaNs and imbalance natively).

In [ ]:
def train_lr(X_tr, y_tr):
    imputer = SimpleImputer(strategy='median')
    X_imp = imputer.fit_transform(X_tr)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)
    model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)
    model.fit(X_scaled, y_tr)
    return model, imputer, scaler

lr_model, lr_imputer, lr_scaler = train_lr(X_train, y_train)
print("Logistic Regression trained.")


In [ ]:
def train_xgb(X_tr, y_tr, params=None):
    pos = y_tr.sum()
    neg = len(y_tr) - pos
    default_params = dict(
        objective='binary:logistic',
        eval_metric='auc',
        max_depth=5,
        learning_rate=0.1,
        n_estimators=300,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(neg / pos if pos > 0 else 1.0),
        random_state=RANDOM_SEED
    )
    params = params or default_params
    model = XGBClassifier(**params)
    model.fit(X_tr, y_tr)
    return model

xgb_model_default = train_xgb(X_train, y_train)
print("XGBoost (default params) trained.")


<a id='8'></a>
## 8. Model Evaluation (on validation set — internal check before final testing)

In [ ]:
NEWS2_MAX = 20  # fixed ceiling for the simplified 5-component score used here

def evaluate_model(y_true, y_score, threshold=0.5, model_name=""):
    auroc = roc_auc_score(y_true, y_score)
    pr_auc = average_precision_score(y_true, y_score)
    y_pred = (y_score >= threshold).astype(int)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    brier = brier_score_loss(y_true, y_score)
    return {
        'model': model_name, 'AUROC': auroc, 'PR_AUC': pr_auc,
        'F1': f1, 'Recall': recall, 'Precision': precision, 'Brier': brier
    }

# NEWS2: normalize by a FIXED ceiling (NEWS2_MAX), not each split's own max —
# using per-split .max() would make the same raw score map to a different probability
# in val vs. test, which is not a valid comparison.
news2_score_val = val_df['news2_score'] / NEWS2_MAX
news2_metrics = evaluate_model(y_val, news2_score_val, threshold=(5 / NEWS2_MAX),
                                model_name='NEWS2')

# Logistic Regression
X_val_imp = lr_imputer.transform(X_val)
X_val_scaled = lr_scaler.transform(X_val_imp)
lr_score_val = lr_model.predict_proba(X_val_scaled)[:, 1]
lr_metrics = evaluate_model(y_val, lr_score_val, model_name='Logistic Regression')

# XGBoost
xgb_score_val = xgb_model_default.predict_proba(X_val)[:, 1]
xgb_metrics = evaluate_model(y_val, xgb_score_val, model_name='XGBoost (default)')

results_df = pd.DataFrame([news2_metrics, lr_metrics, xgb_metrics])
results_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, score in [('NEWS2', news2_score_val), ('LR', lr_score_val), ('XGBoost', xgb_score_val)]:
    fpr, tpr, _ = roc_curve(y_val, score)
    axes[0].plot(fpr, tpr, label=name)
    prec, rec, _ = precision_recall_curve(y_val, score)
    axes[1].plot(rec, prec, label=name)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title("ROC Curve"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend()
axes[1].set_title("Precision-Recall Curve"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
plt.tight_layout()
plt.show()


**Model comparison bar chart and calibration (reliability) plot:**

In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Side-by-side metric comparison bars ---
metrics_to_plot = ['AUROC', 'PR_AUC', 'F1', 'Recall']
plot_df = results_df.set_index('model')[metrics_to_plot]
plot_df.plot(kind='bar', ax=axes[0], color=['#028090', '#00A896', '#02C39A', '#F96167'])
axes[0].set_title("Validation metrics by model")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].tick_params(axis='x', rotation=20)

# --- Calibration / reliability curve ---
for name, score in [('NEWS2', news2_score_val), ('LR', lr_score_val), ('XGBoost', xgb_score_val)]:
    frac_pos, mean_pred = calibration_curve(y_val, score, n_bins=5, strategy='quantile')
    axes[1].plot(mean_pred, frac_pos, marker='o', label=name)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfectly calibrated')
axes[1].set_title("Calibration (reliability) curve")
axes[1].set_xlabel("Mean predicted probability")
axes[1].set_ylabel("Observed fraction positive")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


**Lead time metric** (computed properly on the *test* set in Section 10 — placeholder logic shown here).

In [ ]:
def compute_lead_time(df, y_score, threshold, event_times_by_stay):
    """For each ICU stay with a true death event, find how many hours before the
    ACTUAL event timestamp (not the first positive-labeled window) the model's score
    first crossed threshold. Alerts at or after the event time do not count as
    'before event' warnings — only strictly earlier alerts qualify.

    Correction from review: the earlier version used the first positive-labeled window
    as a proxy for event time, which is wrong when multiple windows are positive in the
    run-up to death; it also allowed alert_time <= event_time, which can count an alert
    fired at the moment of death as a 'warning'. Both are fixed here.

    event_times_by_stay: dict/Series mapping icustay_id -> actual event time (hours since admit).
    """
    df = df.copy()
    df['score'] = y_score
    lead_times = []
    for icustay_id, grp in df.groupby('icustay_id'):
        event_t = event_times_by_stay.get(icustay_id, np.nan)
        if pd.isna(event_t):
            continue  # this stay had no death event
        grp = grp.sort_values('t')
        alerts = grp[(grp['score'] >= threshold) & (grp['t'] < event_t)]
        if not alerts.empty:
            first_alert_t = alerts['t'].min()
            lead_times.append(event_t - first_alert_t)
    return np.array(lead_times)

print("compute_lead_time() defined (uses actual event timestamp, strict '<' before event) "
      "— applied on test set in Section 10.")


<a id='9'></a>
## 9. Hyperparameter Tuning (XGBoost, patient-grouped CV)

In [ ]:
search_space = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 300, 500],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

base_model = XGBClassifier(
    objective='binary:logistic', eval_metric='auc', random_state=RANDOM_SEED,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1)
)

# Pass GroupKFold directly (not a pre-materialized .split() generator) and pass groups=
# explicitly to .fit() — this is the safer, more explicit pattern per review feedback.
search = RandomizedSearchCV(
    base_model, param_distributions=search_space, n_iter=10, scoring='roc_auc',
    cv=GroupKFold(n_splits=3), random_state=RANDOM_SEED, n_jobs=-1
)
search.fit(X_train, y_train, groups=train_df['subject_id'])

print("Best params:", search.best_params_)
print("Best CV AUROC:", search.best_score_)

xgb_model_tuned = search.best_estimator_
xgb_score_val_tuned = xgb_model_tuned.predict_proba(X_val)[:, 1]
tuned_metrics = evaluate_model(y_val, xgb_score_val_tuned, model_name='XGBoost (tuned)')
results_df = pd.concat([results_df, pd.DataFrame([tuned_metrics])], ignore_index=True)
results_df


_If time is tight: tuning is limited to a small 10-iteration random search over a fixed grid;
further tuning could likely improve performance further._

<a id='10'></a>
## 10. Final Testing — **FINAL MODEL, NO FURTHER TUNING AFTER THIS POINT**

Retrain the tuned XGBoost on train+val combined, evaluate once on the held-out test set.

**Correction from review:** the operating threshold must be chosen on the *validation* set,
not hardcoded to 0.5 — a fixed 0.5 threshold has no clinical justification and, with a rare
positive class, will usually under-alert. We select the lowest threshold that achieves at
least 80% recall on validation, then freeze it and apply it unchanged to the test set.

In [ ]:
# --- Select operating threshold on VALIDATION set only ---
TARGET_RECALL = 0.80
thresholds_to_try = np.linspace(0.05, 0.95, 91)
candidates = []
for thresh in thresholds_to_try:
    y_pred_val = (xgb_score_val_tuned >= thresh).astype(int)
    rec = recall_score(y_val, y_pred_val, zero_division=0)
    prec = precision_score(y_val, y_pred_val, zero_division=0)
    f1 = f1_score(y_val, y_pred_val, zero_division=0)
    candidates.append({'thresh': thresh, 'recall': rec, 'precision': prec, 'f1': f1})

cand_df = pd.DataFrame(candidates)
high_recall = cand_df[cand_df['recall'] >= TARGET_RECALL]
if not high_recall.empty:
    best_threshold = float(high_recall.iloc[-1]['thresh'])
else:
    best_threshold = float(cand_df.sort_values('f1', ascending=False).iloc[0]['thresh'])

OPERATING_THRESHOLD = best_threshold
print(f"Selected operating threshold: {OPERATING_THRESHOLD:.3f}")


In [ ]:
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])
trainval_df = pd.concat([train_df, val_df])

final_params = search.best_params_.copy()
final_params.update(dict(
    objective='binary:logistic', eval_metric='auc', random_state=RANDOM_SEED,
    scale_pos_weight=(y_trainval == 0).sum() / max((y_trainval == 1).sum(), 1)
))
xgb_final = XGBClassifier(**final_params)
xgb_final.fit(X_trainval, y_trainval)

xgb_score_test = xgb_final.predict_proba(X_test)[:, 1]
final_metrics = evaluate_model(y_test, xgb_score_test, threshold=OPERATING_THRESHOLD,
                                model_name='XGBoost (FINAL, test set)')

news2_score_test = test_df['news2_score'] / NEWS2_MAX
news2_test_metrics = evaluate_model(y_test, news2_score_test,
                                     threshold=(5 / NEWS2_MAX),
                                     model_name='NEWS2 (test set)')

X_test_imp = lr_imputer.transform(X_test)
X_test_scaled = lr_scaler.transform(X_test_imp)
lr_score_test = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_test_metrics = evaluate_model(y_test, lr_score_test, model_name='Logistic Regression (test set)')

final_results_df = pd.DataFrame([news2_test_metrics, lr_test_metrics, final_metrics])
print("=== FINAL TEST SET RESULTS ===")
final_results_df


**Confusion matrix (final XGBoost, test set) and final metrics comparison:**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Confusion matrix at the operating threshold selected on validation ---
y_pred_test = (xgb_score_test >= OPERATING_THRESHOLD).astype(int)
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted: No event', 'Predicted: Event'],
            yticklabels=['Actual: No event', 'Actual: Event'])
axes[0].set_title(f"Confusion Matrix — XGBoost (threshold={OPERATING_THRESHOLD:.2f})")

# --- Final test-set metric comparison across all 3 models ---
metrics_to_plot = ['AUROC', 'PR_AUC', 'F1', 'Recall']
final_plot_df = final_results_df.set_index('model')[metrics_to_plot]
final_plot_df.plot(kind='bar', ax=axes[1], color=['#028090', '#00A896', '#02C39A', '#F96167'])
axes[1].set_title("Final test-set metrics by model")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].legend(loc='lower right', fontsize=9)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


**XGBoost native feature importance (top 15):**

In [ ]:
importances = pd.Series(xgb_final.feature_importances_, index=feature_cols)
top_features = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
top_features[::-1].plot(kind='barh', ax=ax, color='#028090')
ax.set_title("Top 15 features by XGBoost importance")
ax.set_xlabel("Importance (gain-based)")
plt.tight_layout()
plt.show()


In [ ]:
# Bootstrap 95% CI for final AUROC
n_boot = 1000
boot_aurocs = []
rng = np.random.RandomState(RANDOM_SEED)
test_idx_arr = np.arange(len(y_test))
for _ in range(n_boot):
    sample_idx = rng.choice(test_idx_arr, size=len(test_idx_arr), replace=True)
    y_s = y_test.values[sample_idx]
    score_s = xgb_score_test[sample_idx]
    if len(np.unique(y_s)) < 2:
        continue
    boot_aurocs.append(roc_auc_score(y_s, score_s))

ci_low, ci_high = np.percentile(boot_aurocs, [2.5, 97.5])
print(f"Final XGBoost test AUROC: {final_metrics['AUROC']:.3f} (95% CI: {ci_low:.3f}-{ci_high:.3f})")


In [ ]:
# Build a lookup of actual event (death) time per ICU stay, for the corrected lead-time calc
event_times_by_stay = cohort_stays.set_index('icustay_id')['death_hours_since_admit'].to_dict()

# Lead time on test set, using the threshold selected on validation above (no re-tuning on test)
lead_times = compute_lead_time(test_df, xgb_score_test, OPERATING_THRESHOLD, event_times_by_stay)

if len(lead_times) > 0:
    print(f"Median lead time: {np.median(lead_times):.1f}h "
          f"(IQR: {np.percentile(lead_times, 25):.1f}-{np.percentile(lead_times, 75):.1f}h)")
    print(f"% events warned >=4h ahead: {100*np.mean(lead_times >= 4):.1f}%")
    print(f"% events warned >=6h ahead: {100*np.mean(lead_times >= 6):.1f}%")
else:
    print("No true-positive events with a prior alert found in test set "
          "(expected on the small demo cohort — revisit with full MIMIC-IV).")


**Sample patient risk-over-time plot** — this is the same visual the Streamlit dashboard
will show live: predicted risk score for one ICU stay, hour by hour, with the alert
threshold marked.

In [ ]:
# Pick an example stay: prefer one with a real death event if the test set has one,
# otherwise fall back to the highest-ever-scored stay for illustration.
test_df_scored = test_df.copy()
test_df_scored['score'] = xgb_score_test

stays_with_event = [sid for sid in test_df_scored['icustay_id'].unique()
                     if pd.notna(event_times_by_stay.get(sid, np.nan))]
example_stay_id = stays_with_event[0] if stays_with_event else \
    test_df_scored.loc[test_df_scored['score'].idxmax(), 'icustay_id']

example = test_df_scored[test_df_scored['icustay_id'] == example_stay_id].sort_values('t')
event_t = event_times_by_stay.get(example_stay_id, np.nan)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(example['t'], example['score'], color='#21295C', linewidth=2, label='Predicted risk score')
ax.axhline(OPERATING_THRESHOLD, color='#F96167', linestyle='--', label=f'Alert threshold ({OPERATING_THRESHOLD:.2f})')
if pd.notna(event_t):
    ax.axvline(event_t, color='#990011', linestyle=':', linewidth=2, label='Actual event (death)')
alert_points = example[example['score'] >= OPERATING_THRESHOLD]
ax.scatter(alert_points['t'], alert_points['score'], color='#F96167', zorder=5, s=40, label='Alert raised')

ax.set_title(f"Risk score over time — ICU stay {example_stay_id}")
ax.set_xlabel("Hours since ICU admission")
ax.set_ylabel("Predicted risk of deterioration")
ax.set_ylim(0, 1)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()


<a id='11'></a>
## 11. Explainability (SHAP)

SHAP run on a sampled subset of the test set to keep runtime manageable.

In [ ]:
shap_sample = X_test.sample(min(200, len(X_test)), random_state=RANDOM_SEED)
explainer = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(shap_sample)

shap.summary_plot(shap_values, shap_sample, plot_type="bar", show=True)


In [ ]:
# Example patients: one high risk, one low risk, one borderline
test_scores_series = pd.Series(xgb_score_test, index=X_test.index)
high_idx = test_scores_series.idxmax()
low_idx = test_scores_series.idxmin()
borderline_idx = (test_scores_series - 0.5).abs().idxmin()

for label_str, idx in [("HIGH RISK", high_idx), ("LOW RISK", low_idx), ("BORDERLINE", borderline_idx)]:
    print(f"\n--- {label_str} example (risk score = {test_scores_series[idx]:.3f}) ---")
    single_shap = explainer.shap_values(X_test.loc[[idx]])
    shap.force_plot(explainer.expected_value, single_shap[0], X_test.loc[idx],
                     matplotlib=True, show=True)


<a id='12'></a>
## 12. Export Artifacts

Saved to `drive/MyDrive/icu_early_warning/artifacts/`. These are loaded by the
Streamlit app in `app/dashboard.py`.

In [ ]:
os.makedirs(ARTIFACT_DIR, exist_ok=True)

xgb_final.save_model(os.path.join(ARTIFACT_DIR, 'xgb_final.json'))
joblib.dump(lr_model, os.path.join(ARTIFACT_DIR, 'lr_model.pkl'))
joblib.dump(lr_imputer, os.path.join(ARTIFACT_DIR, 'lr_imputer.pkl'))
joblib.dump(lr_scaler, os.path.join(ARTIFACT_DIR, 'lr_scaler.pkl'))

with open(os.path.join(ARTIFACT_DIR, 'feature_names.json'), 'w') as f:
    json.dump(feature_cols, f, indent=2)

metadata = {
    'horizon_hours': H,
    'window_hours': W,
    'operating_threshold': OPERATING_THRESHOLD,
    'final_test_auroc': float(final_metrics['AUROC']),
    'final_test_auroc_ci': [float(ci_low), float(ci_high)],
    'random_seed': RANDOM_SEED
}
with open(os.path.join(ARTIFACT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

final_results_df.to_csv(os.path.join(ARTIFACT_DIR, 'final_results.csv'), index=False)

print("Artifacts saved to:", ARTIFACT_DIR)
print(os.listdir(ARTIFACT_DIR))


<a id='13'></a>
## 13. Monitoring & Retraining (Design — not implemented, no live data available)

**What we'd monitor in production:**
- Prediction score distribution over time (detect model drift).
- Input feature drift (e.g., new monitoring device, changed measurement protocol).
- Calibration over time — do predicted risks still match observed outcome rates?
- Alert volume and false-alarm rate (clinician trust / alert fatigue).

**Retraining plan:**
- Periodic retraining cadence (e.g., quarterly), triggered earlier if drift metrics
  exceed a threshold.
- A/B test candidate model against the currently deployed model on recent data before
  promoting it.
- Version all models; retain rollback capability to the last known-good version.

**How this would work in a real hospital:** predictions would be surfaced at the bedside
or on a central monitoring board, with alert thresholds set jointly with clinical staff
to balance sensitivity against alert fatigue, and a human always in the loop on any
intervention decision.

<a id='run'></a>
## How to Run

1. Open this notebook in Google Colab.
2. Run Section 0 top to bottom — it will mount Drive and prompt for PhysioNet
   credentials on first run only (subsequent runs skip re-download).
3. Run all remaining sections in order (they depend on variables defined earlier).
4. Section 10 produces the final, frozen model — do not re-tune after this point.
5. Section 12 exports `xgb_final.json`, `lr_model.pkl`, `lr_imputer.pkl`, `lr_scaler.pkl`,
   `feature_names.json`, and `metadata.json` to Drive.
6. The Streamlit app (`app/dashboard.py`, built separately) loads these artifacts directly
   from the same Drive path to power the live demo — no retraining needed at demo time.

**Known limitations (state these honestly in your presentation):**
- Trained on the ~100-patient MIMIC-III Demo cohort — full MIMIC-IV would give more
  robust estimates and is the intended production dataset.
- **Deterioration label is in-hospital death only** in this build. The intended composite
  target (death OR new vasopressor OR new intubation OR cardiac arrest) requires
  `INPUTEVENTS`/`PROCEDUREEVENTS` tables not included in the demo extract — say this
  explicitly rather than implying the full composite was trained.
- NEWS2 here is a **simplified 5-component score** (resp rate, SpO2, temp, SBP, HR) —
  it omits consciousness level and supplemental-O2 scoring present in the official NEWS2,
  so call it "simplified NEWS2-style score," not official NEWS2.
- Measurement-count features carry informative-missingness leakage risk (more lab draws
  can reflect clinical suspicion, not just physiology).
- Patient-level split is stratified on ever-positive outcome, but with ~100 patients the
  positive class is likely very small — report exact split counts, don't just assume balance.
- Offline evaluation only — no live/prospective validation.
- SHAP shows what influenced the model's prediction, not what caused the patient's
  deterioration — keep that distinction explicit in any clinical framing.
